In [1]:
import pickle
import sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
ds = sklearn.datasets.fetch_openml('heart-disease')
print(ds.keys())
print(f"Features: {ds.feature_names}")
df = ds.frame

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])
Features: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [3]:
df.shape

(303, 14)

In [4]:
df.head()
# sex       -> 1 = male, 0 = female
# cp        -> chest pain type (0 = typical angina, 1 = atypical angina, 2 = non-anginal pain/chest pain but not cardiac (muscle, reflux), 3 = asymptomatic/no chest pain)
# trestbps  -> resting blood pressure (mm Hg)
# chol      -> serum cholesterol (mg/dl)
# fbs       -> fasting blood sugar > 120 mg/dl (1/0)
# restecg   -> abnormalities from ECG during rest (0 = normal, 1 = ST dips, T wave is flattened, 2 = enlarged left ventricle produces giant R spike)
# thalach   -> maximum heart rate during exercise (higher is better, lower means heart cannot handle demand)
# exang     -> exercise-induced angina (1/0)
# oldpeak   -> how much ST depression dropped below baseline during exercise (mm)
# slope     -> slope of ST depression during exercise (0 = upsloping/best, 1 = flat/concerning, 2 = downsloping/worst)
# ca        -> number of major vessels colored by fluoroscopy (0-3) (fluoroscopy - X-ray video)
# thal      -> Thalassemia blood disorder measured by stress test (0 = missing, 1 = fixed defect, 2 = normal, 3 = reversible defect)
# target    -> disease (1) or not (0)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,3.0,145.0,233.0,1.0,0.0,150.0,0.0,2.3,0.0,0.0,1.0,1.0
1,37.0,1.0,2.0,130.0,250.0,0.0,1.0,187.0,0.0,3.5,0.0,0.0,2.0,1.0
2,41.0,0.0,1.0,130.0,204.0,0.0,0.0,172.0,0.0,1.4,2.0,0.0,2.0,1.0
3,56.0,1.0,1.0,120.0,236.0,0.0,1.0,178.0,0.0,0.8,2.0,0.0,2.0,1.0
4,57.0,0.0,0.0,120.0,354.0,0.0,1.0,163.0,1.0,0.6,2.0,0.0,2.0,1.0


In [5]:
for i in ds.feature_names:
    print(df[i].value_counts(), end="\n\n")

age
58.0    19
57.0    17
54.0    16
59.0    14
52.0    13
51.0    12
62.0    11
44.0    11
56.0    11
60.0    11
64.0    10
41.0    10
63.0     9
67.0     9
65.0     8
53.0     8
61.0     8
45.0     8
43.0     8
42.0     8
55.0     8
66.0     7
48.0     7
50.0     7
46.0     7
49.0     5
47.0     5
68.0     4
39.0     4
35.0     4
70.0     4
40.0     3
69.0     3
38.0     3
71.0     3
37.0     2
34.0     2
29.0     1
74.0     1
76.0     1
77.0     1
Name: count, dtype: int64

sex
1.0    207
0.0     96
Name: count, dtype: int64

cp
0.0    143
2.0     87
1.0     50
3.0     23
Name: count, dtype: int64

trestbps
120.0    37
130.0    36
140.0    32
110.0    19
150.0    17
138.0    13
128.0    12
125.0    11
160.0    11
112.0     9
132.0     8
118.0     7
108.0     6
124.0     6
135.0     6
145.0     5
152.0     5
134.0     5
170.0     4
100.0     4
122.0     4
105.0     3
180.0     3
136.0     3
142.0     3
126.0     3
115.0     3
148.0     2
146.0     2
144.0     2
178.0     2
94.0      

In [6]:
# Impute missing values
thal_most_frequent = df["thal"].value_counts().idxmax()
df.loc[df["thal"] == 0, "thal"] = thal_most_frequent

ca_most_frequent = df["ca"].value_counts().idxmax()
df.loc[df["ca"] == 4, "ca"] = ca_most_frequent

In [7]:
# Preprocess dataset

# Define groups
binary_cols = ["sex", "fbs", "exang"]
numerical_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_cols = ["cp", "restecg", "slope", "thal", "ca"]
# I initially had "ca" as numerical, but I read that each number represents a different state of vascular blockage, i.e. the intervals between 0/1 or 1/2 are not necessarily the same

# 1) One-hot encode categorical cols and drop 1 column to avoid multicollinearity
pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 2) Split dataset to avoid data leakage from normalization
X = df.drop('target', axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3) Normalize numerical cols
scaler = StandardScaler()
# 3.1) calculate mean and std from train set 3.2) rescale numerical cols to mean 0 and std 1
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
# 3.3) use mean and std from train set
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [8]:
# Save preprocessed dataset
with open("heart_disease_processed.pkl", "wb") as f:
    pickle.dump((X_train, X_test, y_train, y_test), f)